# Phase 3 — Baseline logistic model + lift@top decile

**Cohort:** 12,234 workplaces (D008) — features ≤ 2022, target = serious order in 2023 (D004/D005)  
**Model (D010):** Logistic regression, `class_weight='balanced'`  
**Primary metric (D011):** Lift in top score decile vs 8.0% baseline  
**No random split:** Out-of-time design already separates features (2017–2022) from label (2023).

## Chunk A — Load, sanity check, missing values, feature prep

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path("..").resolve()
OUT = ROOT / "data" / "processed"
TARGET = "target_serious_2023"
ID_COL = "workplace_id"

df = pd.read_csv(OUT / "modeling_table.csv")

# --- sanity checks (Phase 2 assertions) ---
assert df.shape[0] == 12_234, f"Expected 12,234 rows, got {df.shape[0]}"
assert df[ID_COL].is_unique, "workplace_id must be unique at workplace grain"
assert 0.05 < df[TARGET].mean() < 0.15, "Target rate should be ~8%"

baseline_rate = df[TARGET].mean()
print(f"Shape: {df.shape}")
print(f"Target rate: {baseline_rate:.1%}  ({df[TARGET].sum():,} positives)")
print("\nMissing values:")
print(df.isna().sum()[lambda s: s > 0])
print(f"\npct_complied_5y non-null: {df['pct_complied_5y'].notna().sum():,} / {len(df):,}")

In [ ]:
# --- missing-value strategy ---
# primary_naics: 4 rows with no NAICS in history → drop (negligible)
# pct_complied_5y: NaN = no orders with status → add flag + impute to 1.0 (no violation on record)

n_before = len(df)
df = df.dropna(subset=["primary_naics"]).copy()
print(f"Dropped {n_before - len(df)} rows with missing primary_naics")

df["has_order_history"] = df["pct_complied_5y"].notna().astype(int)
df["pct_complied_5y"] = df["pct_complied_5y"].fillna(1.0)

# NAICS 2-digit sector (interpretable; avoids 800+ sparse dummies)
df["naics_sector"] = df["primary_naics"].astype(int).astype(str).str[:2]

# log1p on heavy-tailed count features (D010 note)
COUNT_COLS = [
    "n_visits_5y",
    "n_investigations_5y",
    "n_orders_5y",
    "n_stop_work_5y",
    "n_time_unknown_5y",
    "days_since_last_visit",
]
for col in COUNT_COLS:
    df[f"log1p_{col}"] = np.log1p(df[col])

NUM_COLS = [f"log1p_{c}" for c in COUNT_COLS] + [
    "investigation_ratio",
    "pct_complied_5y",
    "has_order_history",
]
CAT_COLS = ["naics_sector"]
FEATURE_COLS = NUM_COLS + CAT_COLS

X = df[FEATURE_COLS]
y = df[TARGET]
print(f"Modeling rows: {len(df):,}  |  Features: {len(FEATURE_COLS)}")

## Chunk B — Train, score, lift@top decile, coefficients

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUM_COLS),
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False),
            CAT_COLS,
        ),
    ]
)

pipe = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "clf",
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
        ),
    ]
)

pipe.fit(X, y)
proba = pipe.predict_proba(X)[:, 1]
df["score"] = proba

feature_names = NUM_COLS + list(
    pipe.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(CAT_COLS)
)
coefs = pd.Series(pipe.named_steps["clf"].coef_[0], index=feature_names).sort_values()
print("Model fitted on full 2023 evaluation cohort (out-of-time, no random split).")

In [ ]:
def lift_top_decile(y_true, scores, baseline):
    """Rate in top 10% scored vs overall baseline."""
    n_top = int(np.ceil(len(y_true) * 0.10))
    top_idx = np.argsort(-scores)[:n_top]
    top_rate = y_true.iloc[top_idx].mean()
    return top_rate, top_rate / baseline, n_top


top_rate, lift, n_top = lift_top_decile(y, proba, baseline_rate)
pr_auc = average_precision_score(y, proba)
roc_auc = roc_auc_score(y, proba)

print("=" * 50)
print(f"Baseline serious-order rate:  {baseline_rate:.1%}")
print(f"Top-decile rate (n={n_top:,}): {top_rate:.1%}")
print(f"LIFT @ top 10%:               {lift:.2f}×")
print(f"PR-AUC (secondary):           {pr_auc:.3f}")
print(f"ROC-AUC (secondary):          {roc_auc:.3f}")
print("=" * 50)
print(f'\nInterview line: "Top-decile workplaces had a {lift:.1f}× higher')
print(f'serious-order rate than the {baseline_rate:.0%} baseline."')

In [ ]:
# Interpretable drivers — numeric features first (best for Slide 3)
numeric_coefs = coefs[NUM_COLS].sort_values(ascending=False)
print("Top numeric drivers (positive → higher risk):")
print(numeric_coefs.head(5).to_string())
print("\nTop numeric drivers (negative → lower risk):")
print(numeric_coefs.tail(5).to_string())

sector_coefs = coefs[coefs.index.str.startswith("naics_sector_")].sort_values(ascending=False)
print("\nHighest-risk NAICS sectors (2-digit):")
print(sector_coefs.head(3).to_string())
print("\nLowest-risk NAICS sectors (2-digit):")
print(sector_coefs.tail(3).to_string())

## Chunk C — Decile lift chart + save

In [ ]:
df["decile"] = pd.qcut(df["score"], 10, labels=False, duplicates="drop") + 1
decile_stats = (
    df.groupby("decile")
    .agg(rate=(TARGET, "mean"), n=(TARGET, "count"))
    .reset_index()
)
decile_stats["lift"] = decile_stats["rate"] / baseline_rate
decile_stats

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(decile_stats["decile"], decile_stats["lift"], color="steelblue", edgecolor="white")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="Baseline (1.0×)")
ax.set_xlabel("Score decile (1 = lowest risk, 10 = highest)")
ax.set_ylabel(f"Lift vs baseline ({baseline_rate:.1%})")
ax.set_title(f"Lift by score decile — top decile: {lift:.1f}×")
ax.set_xticks(decile_stats["decile"])
ax.legend()
fig.tight_layout()

chart_path = OUT / "lift_chart.png"
fig.savefig(chart_path, dpi=150)
plt.show()
print(f"Saved {chart_path}")